# Hybrid 09 · Final model selection: the deployed grading model

**Objective:** document which model the application serves, exactly how it was built, the evidence behind the choice, and what that evidence does **not** support.

**Decision:** the application grade comes from the **tuned hybrid Random Forest** (24 recorded features + frozen ResNet-50 image embedding, PCA 32). The fitted **ECD tree** is computed for every case and shown beside it as a reference.

> **This is a project decision, not the outcome of the pre-specified rule.** Notebook 13's rule selected the ECD tree (Hybrid 06). The hybrid was chosen afterwards, with that result known (`ANALYSIS_PLAN.md`, Amendment 2). The hybrid has **not** been shown to grade better than the ECD tree or the tabular model.

**Derived from:** `MODEL_SELECTION.md`, `ANALYSIS_PLAN.md` (Amendment 2), `scripts/train_hybrid_grading_model.py`, `ml-service/models/grading_hybrid_rf.{joblib,json}`, `ml-service/pipelines/grading/`.
**Runtime:** about 3 minutes (re-runs the deployment tuning in memory; set `VERIFY_DEPLOYED_TRAINING = False` to skip). Nothing is written or exported.

In [1]:
VERIFY_DEPLOYED_TRAINING = True

Set up paths and load the saved nested-CV fold scores of notebooks 13 and 15 (verified against live re-runs in Hybrid 06 and 08).

In [2]:
import hashlib
import json
import sys
import time
import warnings
from pathlib import Path

GRADING_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "modeling_setup.py").is_file())
sys.path[:0] = [str(GRADING_DIR / "scripts"), str(GRADING_DIR / "notebooks_cleaned")]

import joblib
import numpy as np
import pandas as pd
import shap

import cleaned_setup as cs
import modeling_setup as ms

warnings.filterwarnings("ignore", category=UserWarning)
k3, folds, has_ecd = cs.primary_cohort()
N_TRAIN, N_TEST = cs.mean_fold_sizes(folds, k3["Tissue_ID"])
nested = pd.read_csv(ms.PROCESSED_DIR / "nested_tuning_fold_scores.csv")
extended = pd.read_csv(ms.PROCESSED_DIR / "extended_tuning_fold_scores.csv")

## 1. The evidence: primary nested-CV comparison (notebook 13)

Same 470 K4 tissues and 25 donor-grouped outer folds for every row; fold-level mean ± SD.

In [3]:
MODELS = ["ECD tree", "Tuned tabular", "Tuned hybrid"]
cs.summarise(nested, MODELS, cs.METRICS)

,Macro F1,Balanced accuracy,Accuracy,Recall B
ECD tree,0.796 ± 0.038,0.768 ± 0.043,0.804 ± 0.031,0.790 ± 0.117
Tuned tabular,0.805 ± 0.034,0.781 ± 0.037,0.809 ± 0.030,0.793 ± 0.115
Tuned hybrid,0.793 ± 0.047,0.775 ± 0.050,0.797 ± 0.040,0.789 ± 0.130


Paired macro F1 comparisons (corrected t-test, Holm across T1–T3).

In [4]:
t = cs.paired_comparisons(nested, {"T1": ("Tuned hybrid", "ECD tree"), "T2": ("Tuned tabular", "ECD tree"),
                                   "T3": ("Tuned hybrid", "Tuned tabular")}, cs.METRICS, N_TRAIN, N_TEST)
t[t["Metric"] == "Macro F1"][["ID", "Comparison", "Difference", "CI low", "CI high", "p", "p (Holm)", "Folds improved"]].round(4)

,ID,Comparison,Difference,CI low,CI high,p,p (Holm),Folds improved
0,T1,Tuned hybrid − ECD tree,-0.0025,-0.0341,0.0291,0.8730,1.0,11/25
4,T2,Tuned tabular − ECD tree,0.0091,-0.0102,0.0284,0.3421,1.0,18/25
8,T3,Tuned hybrid − Tuned tabular,-0.0115,-0.0448,0.0217,0.4811,1.0,8/25


Follow-up: the extended hybrid search (notebook 15) against the deployed configuration.

In [5]:
f = cs.paired_comparisons(extended, {"F1": ("Extended RF hybrid", "Tuned RF hybrid")}, cs.METRICS_WITH_KAPPA, N_TRAIN, N_TEST)
display(cs.summarise(extended, ["Tuned RF hybrid", "Extended RF hybrid"], cs.METRICS_WITH_KAPPA))
f[f["Metric"] == "Macro F1"][["Comparison", "Difference", "CI low", "CI high", "p", "Folds improved"]].round(4)

,Macro F1,Balanced accuracy,Accuracy,Recall B,Kappa (quadratic)
Tuned RF hybrid,0.793 ± 0.047,0.775 ± 0.050,0.797 ± 0.040,0.789 ± 0.130,0.764 ± 0.066
Extended RF hybrid,0.802 ± 0.035,0.781 ± 0.041,0.803 ± 0.029,0.793 ± 0.115,0.770 ± 0.052


,Comparison,Difference,CI low,CI high,p,Folds improved
0,Extended RF hybrid − Tuned RF hybrid,0.0085,-0.016,0.033,0.481,13/25


Check the headline figures quoted in the viva against these recomputed values (3-decimal rounding as in `RESULTS.md`).

In [6]:
means = nested.groupby("Model")[["Macro F1", "Accuracy"]].mean()
ext = extended.groupby("Model")[["Macro F1", "Accuracy", "Recall B"]].mean().loc["Extended RF hybrid"]
f1 = f[f["Metric"] == "Macro F1"].iloc[0]
targets = {
    "ECD tree macro F1 / accuracy": ((0.796, 0.804), tuple(means.loc["ECD tree"])),
    "Tuned tabular macro F1 / accuracy": ((0.805, 0.809), tuple(means.loc["Tuned tabular"])),
    "Tuned hybrid macro F1 / accuracy": ((0.793, 0.797), tuple(means.loc["Tuned hybrid"])),
    "Extended hybrid macro F1 / accuracy / B recall": ((0.802, 0.803, 0.793), tuple(ext)),
    "Extended − deployed: difference / CI low / CI high": ((0.008, -0.016, 0.033), (f1["Difference"], f1["CI low"], f1["CI high"])),
    "Extended − deployed: p": ((0.48,), (round(f1["p"], 2),)),
}
for label, (expected, actual) in targets.items():
    ok = all(round(a, 3) == e for e, a in zip(expected, actual))
    print(f"{'OK ' if ok else 'MISMATCH'} {label}: {', '.join(f'{a:.4f}' for a in actual)}")
    assert ok

OK  ECD tree macro F1 / accuracy: 0.7958, 0.8044
OK  Tuned tabular macro F1 / accuracy: 0.8049, 0.8086
OK  Tuned hybrid macro F1 / accuracy: 0.7934, 0.7971
OK  Extended hybrid macro F1 / accuracy / B recall: 0.8018, 0.8029, 0.7931
OK  Extended − deployed: difference / CI low / CI high: 0.0085, -0.0160, 0.0330
OK  Extended − deployed: p: 0.4800


## 2. The pre-specified rule selected the ECD tree; the project then chose the hybrid

| Step | What happened |
|---|---|
| Rule (fixed before notebook 13 ran) | Select a tuned model only if it beats the ECD tree with Holm p < 0.05 and no loss of grade B recall > 0.02; otherwise the ECD tree |
| Outcome | Neither model beat the tree (Holm p = 1.00) → **ECD tree selected** |
| Amendment 2 (after the result was known) | Application grade from the **tuned hybrid**; the ECD tree kept as a reference beside every grade |
| Follow-ups 14 and 15 | No model family and no extended search convincingly outperformed the deployed hybrid → it is **retained** |

## 3. The deployed artefact

Load the metadata exported with the model and verify the model file against its recorded SHA-256 (the service performs the same check at load).

In [7]:
meta = json.loads((cs.DEPLOYED_MODEL_DIR / "grading_hybrid_rf.json").read_text(encoding="utf-8"))
model_path = cs.DEPLOYED_MODEL_DIR / "grading_hybrid_rf.joblib"
digest = hashlib.sha256(model_path.read_bytes()).hexdigest()
print("Model file:", model_path.relative_to(cs.REPO_DIR), "| SHA-256 matches metadata:", digest == meta["model_sha256"])
assert digest == meta["model_sha256"]
for label, value in {
    "Model": meta["model"], "Classes": meta["classes"],
    "Inputs": f"{len(meta['features'])} recorded features + {meta['embedding_columns']}-d image embedding",
    "Setting": meta["setting"], "Inner-CV macro F1 (all K3, used for selection)": meta["inner_cv_macro_f1"],
    "Training cohort": f"{meta['training']['tissues']} tissues, {meta['training']['donors']} donors, {meta['training']['grades']}",
    "Trained on": meta["training"]["trained_on"], "Library versions": meta["training"]["versions"],
}.items():
    print(f"{label}: {value}")

Model file: ml-service\models\grading_hybrid_rf.joblib | SHA-256 matches metadata: True
Model: hybrid_random_forest_24_resnet50
Classes: ['A', 'A+', 'B']
Inputs: 24 recorded features + 2048-d image embedding
Setting: {'pca_components': 32, 'min_samples_leaf': 1, 'max_depth': 15, 'max_features': 0.3, 'class_weight': None, 'n_estimators': 300, 'random_state': 42}
Inner-CV macro F1 (all K3, used for selection): 0.7741
Training cohort: 551 tissues, 312 donors, {'A+': 277, 'A': 196, 'B': 78}
Trained on: 2026-10-16
Library versions: {'scikit_learn': '1.9.1', 'pandas': '3.0.6', 'numpy': '2.4.6'}


The model was trained on exactly the data versions used throughout these notebooks.

In [8]:
print("Dataset SHA-256 matches:", hashlib.sha256(ms.DATA_PATH.read_bytes()).hexdigest() == meta["training"]["data_sha256"])
print("Embedding cache SHA-256 matches:", hashlib.sha256(cs.EMBEDDING_PATH.read_bytes()).hexdigest() == meta["training"]["embeddings_sha256"])

Dataset SHA-256 matches: True
Embedding cache SHA-256 matches: True


Inspect the fitted pipeline: the tabular branch, the image branch (standardise → PCA 32) and the Random Forest.

In [9]:
pipeline = joblib.load(model_path)
features, forest = pipeline.named_steps["features"], pipeline.named_steps["classifier"]
print([(name, type(t).__name__, len(cols)) for name, t, cols in features.transformers])
print("Image branch:", features.named_transformers_["image"])
print("Forest:", {k: forest.get_params()[k] for k in ["n_estimators", "max_depth", "max_features", "min_samples_leaf", "class_weight", "random_state"]})
print("Model inputs after preprocessing:", forest.n_features_in_)

[('tabular', 'ColumnTransformer', 24), ('image', 'Pipeline', 2048)]
Image branch: Pipeline(steps=[('scaler', StandardScaler()),
                ('pca', PCA(n_components=32, random_state=42))])
Forest: {'n_estimators': 300, 'max_depth': 15, 'max_features': 0.3, 'min_samples_leaf': 1, 'class_weight': None, 'random_state': 42}
Model inputs after preprocessing: 96


Image representation and backbone, as recorded with the model (the service runs the same ResNet-50 exported to ONNX and checks it against a recorded reference output).

In [10]:
backbone = meta["backbone"]
for key in ["source", "preprocessing", "training_image_size"]:
    print(f"{key}: {backbone[key]}")
print("parity with research embeddings:", {k: v for k, v in backbone["parity_with_training_embeddings"].items() if k != "note"})

source: torchvision ResNet-50, IMAGENET1K_V2, fc removed (scripts/export_grading_backbone.py)
preprocessing: 8-bit greyscale at full resolution, /255, repeated to 3 channels, ImageNet mean/std normalisation (notebook 07)
training_image_size: [1296, 972]
parity with research embeddings: {'tissues': 551, 'max_relative_l2_difference': 0.00373, 'same_prediction': 551, 'max_probability_difference': 0.06}


## 4. Reproduce how the deployed model was trained

`train_hybrid_grading_model.py` reruns notebook 13's hybrid grid (PCA {8, 16, 32} × 24 forest settings) with 3-fold donor-grouped CV on **all 551 K3 tissues**, then refits the winner. Run its own functions here (no export) and compare with the deployed file.

In [11]:
import train_hybrid_grading_model as deploy

if VERIFY_DEPLOYED_TRAINING:
    start = time.time()
    cohort = deploy.load_cohort()
    cached = np.load(cs.EMBEDDING_PATH, allow_pickle=True)
    row_of = {t: i for i, t in enumerate(cached["tissue_ids"])}
    Xd = deploy.with_embeddings(cohort[ms.ALL_FEATURES], cached["embeddings"][[row_of[t] for t in cohort["Tissue_ID"]]])
    yd = cohort[ms.TARGET].to_numpy()
    (k, setting), inner_f1 = deploy.tune(Xd, yd, cohort["Donor_Group"].to_numpy())
    refit = deploy.make_pipeline(k, setting).fit(Xd, yd)
    print(f"Selected: PCA {k}, {setting}; inner-CV macro F1 {inner_f1:.4f} ({(time.time() - start) / 60:.1f} min)")
    print("Same setting as deployed:", {"pca_components": k, **setting} == {key: meta["setting"][key] for key in ["pca_components", *setting]})
    print("Same predictions as the deployed model on all 551 tissues:", np.array_equal(refit.predict(Xd), pipeline.predict(Xd)),
          f"| max vote-share difference {np.abs(refit.predict_proba(Xd) - pipeline.predict_proba(Xd)).max():.1e}")

Selected: PCA 32, {'min_samples_leaf': 1, 'max_depth': 15, 'max_features': 0.3, 'class_weight': None}; inner-CV macro F1 0.7741 (2.2 min)
Same setting as deployed: True


Same predictions as the deployed model on all 551 tissues: True | max vote-share difference 2.2e-16


The performance estimate for this configuration is notebook 13's nested result (macro F1 0.793 ± 0.047), **not** the inner-CV score above, which was used to choose the setting.

In [12]:
for key, value in meta["evaluation"].items():
    print(f"{key}: {value}")

source: 13_nested_tuning_model_selection.ipynb (25 donor-grouped outer folds, 3-fold inner tuning)
macro_f1: 0.793 ± 0.047
balanced_accuracy: 0.775 ± 0.050
accuracy: 0.797 ± 0.040
grade_b_recall: 0.789 ± 0.130
vs_ecd_tree_macro_f1: −0.003 [−0.034, +0.029], Holm p = 1.00 (not significant)
vs_tuned_tabular_macro_f1: −0.012 [−0.045, +0.022], Holm p = 1.00 (not significant)


## 5. How the application uses it

| Step | Implementation |
|---|---|
| Load | `ml-service/pipelines/grading/hybrid.py`: model checked against its SHA-256; ONNX ResNet-50 checked against its verified record; fails closed |
| Image → embedding | 8-bit greyscale at 1296 × 972, ÷255, 3 channels, ImageNet normalisation → 2048-d (other sizes are resized and flagged for review) |
| Grade | `predict_proba` of this pipeline; inputs not recorded are passed as missing and handled as in training (never 0 or "No"); output = grade + share of trees voting (uncalibrated, not a confidence) |
| Explanation | `explain.py`: TreeSHAP on the forest, grouped back to the 24 inputs + one image contribution, and a SHAP-weighted image map |
| Reference | `service.py`: the ECD tree (`grading_ecd_tree.json`) shown beside the grade |

The ECD reference tree as exported for the service:

In [13]:
spec = json.loads((cs.DEPLOYED_MODEL_DIR / "grading_ecd_tree.json").read_text(encoding="utf-8"))
pd.DataFrame([{"ECD band": f"{b['lower_exclusive'] or '−∞'} < ECD ≤ {b['upper_inclusive'] or '∞'}", "Tree grade": b["grade"], **b["training_counts"]}
              for b in spec["bands"]])

,ECD band,Tree grade,A+,A,B
0,−∞ < ECD ≤ 2232.0,B,0,0,53
1,2232.0 < ECD ≤ 2497.0,A,2,84,8
2,2497.0 < ECD ≤ ∞,A+,242,75,6


Illustration on one K4 tissue where the hybrid and the ECD reference disagree (an in-sample training tissue: this shows the mechanics, it is **not** an evaluation). Grouped TreeSHAP values plus the baseline add up to the hybrid's vote share for its grade.

In [14]:
X3, y3, _ = cs.feature_matrix(k3)
proba = pipeline.predict_proba(X3)
grade = pipeline.classes_[proba.argmax(axis=1)]
tree_grade = np.full(len(k3), None, dtype=object)
tree_grade[has_ecd] = np.select([X3.loc[has_ecd, cs.ECD] <= spec["cut_points"][0], X3.loc[has_ecd, cs.ECD] <= spec["cut_points"][1]], ["B", "A"], "A+")
i = int(np.flatnonzero(has_ecd & (grade != tree_grade))[0])

Z = features.transform(X3.iloc[[i]])
explainer = shap.TreeExplainer(forest, feature_perturbation="tree_path_dependent")
values = np.asarray(explainer.shap_values(Z))[0, :, list(pipeline.classes_).index(grade[i])]
columns = features.get_feature_names_out()
source = ["Image (32 PCA components)" if c.startswith("image__") else
          max((f for f in ms.ALL_FEATURES if c.split("__", 2)[2] == f or c.split("__", 2)[2].startswith(f + "_")), key=len) for c in columns]
grouped = pd.Series(values, index=source).groupby(level=0).sum().sort_values(key=np.abs, ascending=False)
base = np.asarray(explainer.expected_value)[list(pipeline.classes_).index(grade[i])]
print(f"ECD {X3.at[i, cs.ECD]:.0f} → ECD reference {tree_grade[i]} | recorded grade {y3[i]} | hybrid grade {grade[i]}, "
      f"vote shares {dict(zip(map(str, pipeline.classes_), proba[i].round(3).tolist()))}")
print(f"Baseline {base:.3f} + contributions {grouped.sum():+.3f} = {base + grouped.sum():.3f} (vote share for {grade[i]}: {proba[i].max():.3f})")
grouped.head(6).round(3).to_frame(f"Contribution to the vote share for {grade[i]}")

ECD 2538 → ECD reference A+ | recorded grade A | hybrid grade A, vote shares {'A': 0.761, 'A+': 0.219, 'B': 0.02}
Baseline 0.354 + contributions +0.407 = 0.761 (vote share for A: 0.761)


,Contribution to the vote share for A
Image (32 PCA components),0.301
Total_Area_um2,0.079
Total_Cell_Count,0.017
SD,0.008
HEX_percent,0.003
CV,-0.003


## 6. Why the hybrid, and what each reason does not show

| Reason | Limitation |
|---|---|
| It uses all three sources the specialist uses: endothelial morphology, slit-lamp findings and the image (the project's multimodal objective) | Adding the image never significantly improved grading in a valid comparison (Hybrid 03, 04, 06); the one significant result (C3, Hybrid 03) is a masking artefact with lower grade B recall |
| It can depart from the ECD thresholds; shown beside the ECD tree, a disagreement prompts a closer look | Its departures were right about as often as wrong (131 recovered vs 151 wrong, Hybrid 05) |
| It gives a per-case explanation over all inputs (TreeSHAP + image map) | Explanations describe the model, not the cornea; the image map's faithfulness is unevaluated |
| It still produces a suggestion when ECD is missing (15% of K3), flagged for review | Performance there is poor (macro F1 0.47, grade B recall 0.16, Hybrid 04) |

**Why not the alternatives:** the ECD tree (selected by the rule) only restates two cut points, cannot grade tissues without ECD and leaves nothing to explain beyond the threshold, so it is kept as the reference. The tuned tabular model is numerically best (+0.012 over the hybrid, not significant) and simpler, but drops the image and its explanation. **On performance and simplicity alone, the ECD tree or the tabular model would be the choice;** the hybrid was chosen deliberately for the multimodal and explainability objectives.

**Not supported, and not claimed:** that the hybrid grades better than ECD alone or the tabular model (T1 −0.003, T3 −0.012, Holm p = 1.00); that the further tuning (+0.008, p = 0.48) is an improvement; that vote shares are calibrated confidences; any clinical validity.

## Conclusion, limitations and next steps

- All three candidates grade equally well within the precision of this dataset (≈0.79–0.81 macro F1, ≈0.80 accuracy), and no model beat the two-threshold ECD tree. The hybrid is a design choice for multimodal decision support, not a demonstrated gain in accuracy.
- **Limitations:** the choice was made after the rule's result was known; the outer folds had informed earlier decisions, so the 0.793 estimate is not untouched; donors are inferred; image features are generic ImageNet features; no external, prospective or other-microscope validation; vote shares are uncalibrated.
- **What would change the decision:** an evaluation on new, independently graded tissues showing the hybrid underperforms by a clinically relevant margin agreed in advance, or specialist review finding the explanations unhelpful.
- **Next steps:** validation on new tissues (notebook 09, Section 9 plan); a pre-specified search above `max_features` 0.7 evaluated on new tissues; microscopy-specific image representations; a model-free comparison of tissues graded below the ECD thresholds.